[Feature](https://sp.ukdataservice.ac.uk/doc/6926/mrdoc/pdf/6926userguide.pdf)

## PySpark Pipeline

Apache Spark is used for the data-intensive phases of this notebook:

| Phase | Tool | Why |
|-------|------|-----|
| Load & union 3 CSV files (~1.5 M rows) | PySpark | Faster than pandas concat at this scale |
| Missing value analysis | PySpark | Single distributed pass over full dataset |
| Outlier detection | PySpark | `approxQuantile` avoids full sort |
| Temporal EDA (day/year/hour) | PySpark | `groupBy` → tiny pandas frame for plotting |

After this section the data is handed to **pandas** for the geopandas spatial join (LSOA polygon lookup) and all subsequent EDA and modelling.

In [ ]:
!pip install -q pyspark
!pip install reverse_geocoder
!pip install kneed

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 23.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for reverse_geocoder: filename=reverse_geocoder-1.5.1-py3-none-any.whl size=2268067 sha256=dcf1643968697684dce8d18bbb925b01a172e7283b685f0044ebf17ddb02d060
  Stored in directory: /root/.cache/pip/wheels/11/e1/67/6e47f0ad41ea1843d37e1fbe79c6074744a1f4aace641cf800
Successfully built reverse_geocoder


In [ ]:
# Standard library
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patheffects import withStroke
import seaborn as sns

# Geospatial
import geopandas as gpd
from shapely.geometry import Point
import reverse_geocoder as rg

from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from kneed import KneeLocator

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from pyspark.sql.functions import (
    col, when, count,
    sum as spark_sum,
    hour as spark_hour,
    to_timestamp,
)

# Style
plt.style.use('ggplot')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


spark = (SparkSession.builder
    .appName('UK Road Accidents')
    .config('spark.driver.memory', '4g')
    .getOrCreate())

spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready')

Spark 4.0.2 ready


## 1. Data Loading

Three CSV files covering 2005–2007, 2009–2011, and 2012–2014 are read in parallel and unioned into one Spark DataFrame with `unionByName`.

`LSOA_of_Accident_Location` is explicitly cast to `StringType` because `inferSchema` reads it as an integer, which would drop the leading-zero district codes.

In [ ]:
BASE = '/content/drive/MyDrive/UK_accident'

sdf_5to7   = spark.read.csv(f'{BASE}/accidents_2005_to_2007.csv', header=True, inferSchema=True)
sdf_9to11  = spark.read.csv(f'{BASE}/accidents_2009_to_2011.csv', header=True, inferSchema=True)
sdf_12to14 = spark.read.csv(f'{BASE}/accidents_2012_to_2014.csv', header=True, inferSchema=True)

lsoa = 'LSOA_of_Accident_Location'
sdf_5to7   = sdf_5to7.withColumn(lsoa,   col(lsoa).cast(StringType()))
sdf_9to11  = sdf_9to11.withColumn(lsoa,  col(lsoa).cast(StringType()))
sdf_12to14 = sdf_12to14.withColumn(lsoa, col(lsoa).cast(StringType()))

sdf_5to7.show(5)

In [ ]:
assert sdf_5to7.columns == sdf_9to11.columns == sdf_12to14.columns, 'Column mismatch!'
print(f'All files share the same {len(sdf_5to7.columns)} columns: OK')

In [ ]:
sdf = sdf_5to7.unionByName(sdf_9to11).unionByName(sdf_12to14)
print(f'Total rows: {sdf.count():,}')

## 2. Schema Inspection

`printSchema()` confirms the types Spark inferred from the CSVs. Check that numeric columns (speed limit, casualties, vehicles) are `IntegerType` or `DoubleType`, not `StringType` — a common issue when missing values are encoded as empty strings or spaces in the raw file.

In [ ]:
sdf.printSchema()

## 3. Missing Value Analysis

### Overview

A single Spark pass counts nulls for every column using `count(when(col.isNull(), col))`.

In [ ]:

null_counts = sdf.select([count(when(col(c).isNull(), c)).alias(c) for c in sdf.columns])
null_counts.show(vertical=True)

### Per-feature Value Distribution

For columns with nulls, inspecting the full value distribution (including null counts) reveals *why* values are missing before we decide how to impute:

- A column dominated by one value with a high null share likely represents **absence** (e.g., no junction control) → fill `'None'`
- A column with < 1% nulls spread across values → **drop rows** (random missingness)
- A column where nulls have a plausible 'unknown' interpretation → fill `'Unknown'`

In [ ]:

feat_to_check = [
    'Junction_Control',
    'Pedestrian_Crossing-Human_Control',
    'Pedestrian_Crossing-Physical_Facilities',
    'Weather_Conditions',
    'Road_Surface_Conditions',
    'Special_Conditions_at_Site',
    'Carriageway_Hazards',
    'Did_Police_Officer_Attend_Scene_of_Accident',
    'LSOA_of_Accident_Location',
]

for feat in feat_to_check:
    print('\n' + '=' * 40)
    print(f'Feature: {feat}')
    print('-' * 40)
    # Backtick-quote column names that contain hyphens
    safe = f'`{feat}`' if '-' in feat else feat
    (sdf.groupBy(col(safe).alias(feat))
        .count()
        .orderBy('count', ascending=False)
        .show(truncate=False))

### Location Column Consistency

The four location columns — `Location_Easting_OSGR`, `Location_Northing_OSGR`, `Longitude`, `Latitude` — should always be null *together*. A row missing coordinates is unplaceable regardless of which column is checked, so if any one is null the whole row should be dropped. The check below confirms there are no *partial* location nulls that would indicate a data quality issue worth investigating separately.

In [ ]:
loc_cols = ['Location_Easting_OSGR', 'Location_Northing_OSGR', 'Longitude', 'Latitude']

# Count rows where nulls are inconsistent across the four location columns
inconsistent = sdf.filter(
    col('Longitude').isNull() != col('Latitude').isNull()
).count()

null_lat = sdf.filter(col('Latitude').isNull()).count()

print(f'Rows with null Latitude:            {null_lat:,}')
print(f'Rows with inconsistent loc nulls:   {inconsistent:,}')
print(f'All location nulls are consistent:  {inconsistent == 0}')

### Imputation Strategy

| Column(s) | Action | Reason |
|-----------|--------|--------|
| `Junction_Control`, `Special_Conditions_at_Site`, `Carriageway_Hazards` | Fill `'None'` | High null share — null means the feature is absent |
| `Pedestrian_Crossing-*`, `Weather_Conditions`, `Road_Surface_Conditions`, `Time`, `Latitude` | **Drop rows** | < 1 % missing — random missingness, safe to remove |
| `Did_Police_Officer_Attend_Scene_of_Accident`, `LSOA_of_Accident_Location` | Fill `'Unknown'` | Moderate proportion, meaningful unknown category |
| `Junction_Detail` | **Drop column** | > 50 % null, not used in analysis |

In [ ]:
sdf = sdf.drop('Junction_Detail')

fill_none_feat = ['Junction_Control', 'Special_Conditions_at_Site', 'Carriageway_Hazards']
drop_na_feat   = [
    'Pedestrian_Crossing-Human_Control', 'Pedestrian_Crossing-Physical_Facilities',
    'Weather_Conditions', 'Road_Surface_Conditions', 'Time', 'Latitude',
]
fill_unknown_feat = ['Did_Police_Officer_Attend_Scene_of_Accident', 'LSOA_of_Accident_Location']

sdf = sdf.fillna({f: 'None'    for f in fill_none_feat})
sdf = sdf.dropna(subset=drop_na_feat)
sdf = sdf.fillna({f: 'Unknown' for f in fill_unknown_feat})

print(f'Rows after cleaning: {sdf.count():,}')

In [ ]:
null_after = sdf.select([count(when(col(c).isNull(), c)).alias(c) for c in sdf.columns])
null_after.show(vertical=True)

## 4. Outlier Analysis

Outliers are flagged using the **IQR method**: a value is an outlier if it falls below `Q1 − 1.5 × IQR` or above `Q3 + 1.5 × IQR`.

**No rows are removed** — road accident data contains genuinely extreme but valid events (e.g., multi-vehicle pile-ups). The analysis is for awareness and to inform later scaling decisions.


In [ ]:

numeric_cols = ['Number_of_Vehicles', 'Number_of_Casualties', 'Speed_limit']
quantile_probs = [0.01, 0.25, 0.50, 0.75, 0.99]

# Single distributed pass for all columns
q_results = sdf.approxQuantile(numeric_cols, quantile_probs, 0.01)

fences = {}
for c, q_vals in zip(numeric_cols, q_results):
    q1, q3 = q_vals[1], q_vals[3]
    iqr = q3 - q1
    fences[c] = dict(q_vals=q_vals, q1=q1, q3=q3, iqr=iqr,
                     lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)

# Count outliers in one Spark pass
outlier_row = sdf.select([
    spark_sum(
        when((col(c) < fences[c]['lower']) | (col(c) > fences[c]['upper']), 1).otherwise(0)
    ).alias(c)
    for c in numeric_cols
]).collect()[0]

total = sdf.count()

print('Quantile / Outlier Analysis  (IQR method — no removal)')
print('=' * 60)
for c in numeric_cols:
    f = fences[c]
    q = f['q_vals']
    n = outlier_row[c]
    print(f'\n{c}')
    print(f'  P1={q[0]:.1f}  Q1={f["q1"]:.1f}  Median={q[2]:.1f}  Q3={f["q3"]:.1f}  P99={q[4]:.1f}')
    print(f'  IQR={f["iqr"]:.1f}   Whiskers: [{f["lower"]:.1f}, {f["upper"]:.1f}]')
    print(f'  Flagged: {n:,}  ({n / total * 100:.2f} %)')


### Number of Vehicles
The interquartile range is just 1 (Q1=1, Q3=2), meaning most accidents involve
1-2 vehicles. The 2.27% of records flagged beyond the upper whisker of 3.5
represent genuine multi-vehicle pile-ups rather than data errors, so no removal
is warranted.

### Number of Casualties
This means any accident with 2 or more casualties is mechanically flagged as an outlier, which explains the nonsensical 23.3% rate.
This is a known limitation of IQR on heavily right-skewed count data and should not be interpreted as evidence of data quality issues.

### Speed Limit
Zero outliers are flagged because speed limits in Great Britain are a fixed discrete set of regulatory values (20, 30, 40, 50, 60, 70 mph). IQR outlier
detection is not meaningful for this column and it should be treated as a
**categorical variable** downstream.

In [ ]:

fig, axes = plt.subplots(1, len(numeric_cols), figsize=(14, 4))
for ax, c in zip(axes, numeric_cols):
    # 5 % sample — avoids pulling 1.5 M rows to the driver just for a plot
    pdf = sdf.select(c).sample(fraction=0.05, seed=42).toPandas()
    ax.boxplot(pdf[c].dropna(), patch_artist=True,
               boxprops=dict(facecolor='#457B9D', alpha=0.7),
               medianprops=dict(color='#E63946', linewidth=2),
               flierprops=dict(marker='o', markersize=2, alpha=0.3))
    ax.set_title(c.replace('_', ' '), fontsize=10, fontweight='bold')
    ax.set_ylabel('Value')

plt.suptitle('Outlier Distribution — 5% sample (IQR whiskers)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Temporal EDA

Aggregate counts are computed with Spark `groupBy` and pulled to tiny pandas frames (< 25 rows each) for seaborn plotting. Only the aggregated frame — not the full 1.5 M rows — is transferred to the driver.

In [ ]:


sdf = sdf.withColumn('Time', to_timestamp(col('Time'), 'HH:mm'))
sdf = sdf.withColumn('Hour', spark_hour(col('Time')))

In [ ]:
dow_pdf = sdf.groupBy('Day_of_Week').count().orderBy('Day_of_Week').toPandas()
sns.barplot(data=dow_pdf, x='Day_of_Week', y='count')
plt.show()


In [ ]:
year_pdf = sdf.groupBy('Year').count().orderBy('Year').toPandas()
sns.barplot(data=year_pdf, x='Year', y='count', palette='viridis', hue='Year', legend=False)
plt.show()

In [ ]:
hour_pdf = sdf.groupBy('Hour').count().orderBy('Hour').toPandas()
sns.barplot(data=hour_pdf, x='Hour', y='count', palette='viridis', hue='Hour', legend=False)
plt.show()

In [ ]:
sdf.select('LSOA_of_Accident_Location').distinct().show(10)

In [ ]:
df = sdf.toPandas()
spark.stop()
print(f"Converted to pandas: {df.shape}")

In [ ]:


districts = gpd.read_file(f'/content/drive/MyDrive/UK_accident/lsoa.geojson')

geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
gdf_accidents = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

districts = districts.to_crs("EPSG:4326")

df_mapped = gpd.sjoin(gdf_accidents, districts, how="left", predicate="within")

print(df_mapped.head())

In [ ]:
df_mapped.isnull().sum()


In [ ]:
df = df_mapped

In [ ]:

# Batch lookup - much faster than row-by-row
coords = list(zip(df['Latitude'], df['Longitude']))
results = rg.search(coords)

df['region']         = [r['admin2'] for r in results]
df['greater_region'] = [r['admin1'] for r in results]
print(df['greater_region'].value_counts())

In [ ]:
df['greater_region'].value_counts().plot.pie(
    autopct='%1.1f%%',
    figsize=(8, 8),
    colors=sns.color_palette('viridis', df['greater_region'].nunique())
)
plt.ylabel('')
plt.title('Accidents by Region')
plt.show()

In [ ]:
df.info()

In [ ]:
region_lookup = pd.read_csv(f'{BASE}/lad_to_region_en.csv')
region_lookup.info()

Here's how we link everything:

1. **Local_Authority_Districts_Dec_2016.geojson**:
   - Convert `longtitude` and `latitude` to `lad16cd`

2. **[Uk Gov Look up](https://geoportal.statistics.gov.uk/datasets/ons::local-authority-district-to-region-december-2016-lookup-in-en/about)**:
   - Convert `lad16cd` to `rgn16NM`

In [ ]:
region_lookup = region_lookup[['LAD16CD', 'RGN16NM']].drop_duplicates()

df = df.merge(
    region_lookup,
    left_on='lad16cd',
    right_on='LAD16CD',
    how='left'
)

print(df['RGN16NM'].value_counts())
print(f"\nTotal regions: {df['RGN16NM'].nunique()}")


In [ ]:
df.isnull().sum()

In [ ]:
missing_mask = df['RGN16NM'].isna()
print(df[missing_mask]['lad16cd'].str[:3].value_counts())

In [ ]:
df['RGN16NM'] = df['RGN16NM'].fillna(df['greater_region'])

# Sanity check - should be 0
print(df['RGN16NM'].isna().sum())

In [ ]:
region_counts = df['RGN16NM'].value_counts()


In [ ]:
region_counts = df['RGN16NM'].value_counts()

plt.figure(figsize=(10, 10))
plt.pie(
    region_counts,
    labels=region_counts.index,
    autopct='%1.1f%%',
    colors=sns.color_palette('viridis', len(region_counts))
)
plt.title('Accidents by Region')
plt.show()

Geojson for regions: [Kaggle](https://www.kaggle.com/datasets/dorianlazar/uk-regions-geojson)

In [ ]:
regions_geo = gpd.read_file(f'{BASE}/uk_regions.geojson')
print(regions_geo['rgn19nm'].unique())  # adjust col name after seeing columns

In [ ]:
regions_geo['rgn19nm'] = regions_geo['rgn19nm'].replace({
    'East': 'East of England',
    'Yorkshire and the Humber': 'Yorkshire and The Humber'
})

In [ ]:
region_counts = df['RGN16NM'].value_counts().reset_index()
region_counts.columns = ['RGN16NM', 'accident_count']

regions_geo = regions_geo.merge(
    region_counts,
    left_on='rgn19nm',
    right_on='RGN16NM',
    how='left'
)

# Sanity check - should be no NaN in accident_count
print(regions_geo[['rgn19nm', 'accident_count']])

In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(8, 10))
regions_geo.plot(
    column='accident_count',
    ax=ax,
    legend=True,
    cmap='YlOrRd',
    edgecolor='white',
    linewidth=0.5,
    vmin=0,                                   # anchor 0 to the bottom of the scale
    vmax=regions_geo['accident_count'].max(),
    legend_kwds={'label': 'Number of Accidents'}
)

# Labels with outline effect for visibility
for _, row in regions_geo.iterrows():
    centroid = row['geometry'].centroid
    ax.annotate(
        row['rgn19nm'],
        xy=(centroid.x, centroid.y),
        ha='center', fontsize=7, fontweight='bold', color='white',
        path_effects=[withStroke(linewidth=2, foreground='black')]  # outline
    )

ax.set_title('Accidents by Region in UK (2005-2015)', fontsize=14)
ax.axis('off')
plt.tight_layout()

In [ ]:
sns.countplot(df, hue = 'Accident_Severity', x = 'Year', palette= 'viridis')
plt.legend(bbox_to_anchor = [1.05, 1])
plt.show()

In [ ]:
df['Accident_Severity_Label'] = df['Accident_Severity'].map({1: 'Fatal', 2: 'Serious', 3: 'Slight'})


In [ ]:
df[df['RGN16NM'] == 'London']['Urban_or_Rural_Area'].value_counts()

In [ ]:
df['2nd_Road_Class'].unique()

In [ ]:
df['1st_Road_Class'].unique()

In [ ]:
df['Urban_or_Rural_Area_Label'] = df['Urban_or_Rural_Area'].map({1: 'Urban', 2: 'Rural'})

road_class_map = {-1: 'No second road', 1: 'Motorway', 2: 'A(M)', 3: 'A', 4: 'B', 5: 'C', 6: 'Unclassified'}

df['1st_Road_Class_Label'] = df['1st_Road_Class'].map(road_class_map)
df['2nd_Road_Class_Label'] = df['2nd_Road_Class'].map(road_class_map)

In [ ]:

features = ['Junction_Control', 'Speed_limit', 'Light_Conditions', 'Road_Type',
            'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area_Label',
            '1st_Road_Class_Label', '2nd_Road_Class_Label'
            ]

severity_order = ['Fatal', 'Serious', 'Slight']
palette = {'Fatal': '#d62728', 'Serious': '#ff7f0e', 'Slight': '#2ca02c'}

road_order = ['Motorway', 'A(M)', 'A', 'B', 'C', 'Unclassified', 'No second road']

fig, axes = plt.subplots(3, 3, figsize=(16, 18))
axes = axes.flatten()



for i, col in enumerate(features):
    ax = axes[i]

    order = df[col].value_counts().index.tolist()
    order = road_order if 'Road_Class' in col else df[col].value_counts().index.tolist()
    sns.countplot(
        data=df,
        x=col,
        hue='Accident_Severity_Label',
        hue_order=severity_order,
        palette=palette,
        order=order,
        ax=ax
    )
    ax.set_title(col.replace('_', ' '), fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=90)
    ax.legend(title='Severity', loc='upper right')


plt.suptitle('Accident Severity by Road Conditions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
df.info()

In [ ]:
cols_to_drop = [
    # Coordinate duplicates
    "Location_Easting_OSGR",
    "Location_Northing_OSGR",
    "bng_e",
    "bng_n",
    "long",
    "lat",
    # Spatial join artifacts
    "index_right",
    "objectid",
    "st_areashape",
    "st_lengthshape",
    # Duplicate admin columns
    "lad16nmw",  # Welsh name, duplicate of lad16nm
    "LAD16CD",   # incomplete (1.34M) duplicate of lad16cd (1.5M, fully complete)

    # geospartial
    'geometry',

]

df_accident = df.drop(columns=cols_to_drop).dropna().copy()
df_accident.isnull().sum()

In [ ]:
df_accident['country'] = df_accident['greater_region']
df_accident['greater_region'] = df_accident['RGN16NM']
df_accident = df_accident.drop(columns=['RGN16NM'])

In [ ]:
df_accident['greater_region'].unique()

In [ ]:
df_accident["Month"] = pd.to_datetime(df_accident["Date"], dayfirst=True).dt.month


In [ ]:

# Cyclical encode circular time features
for col, period in [("Hour", 24), ("Day_of_Week", 7), ("Month", 12)]:
    df_accident[f"{col}_sin"] = np.sin(2 * np.pi * df_accident[col] / period)
    df_accident[f"{col}_cos"] = np.cos(2 * np.pi * df_accident[col] / period)


In [ ]:
df_accident.info()

In [ ]:
# Run this to check
cols_to_check = [
    "Light_Conditions",
    "Weather_Conditions",
    "Road_Surface_Conditions",
    "Special_Conditions_at_Site",
    "Carriageway_Hazards",
    "Junction_Control",
    "Pedestrian_Crossing-Human_Control",
    "Pedestrian_Crossing-Physical_Facilities",
    "Did_Police_Officer_Attend_Scene_of_Accident",
    "Road_Type",
]

for col in cols_to_check:
    print('\n' + '-' * 50)
    print(f"{col}: {df_accident[col].unique()}")

In [ ]:
light_map = {
    'Daylight: Street light present': 1,
    'Daylight: No street lighting': 2,
    'Daylight: Street lighting unknown': 3,
    'Darkness: Street lights present and lit': 4,
    'Darkness: Street lights present but unlit': 5,
    'Darkness: No street lighting': 6,
    'Darkeness: No street lighting': 6,         # typo in data
    'Darkness: Street lighting unknown': 7,
}

weather_map = {
    'Fine without high winds': 1,
    'Raining without high winds': 2,
    'Snowing without high winds': 3,
    'Fine with high winds': 4,
    'Raining with high winds': 5,
    'Snowing with high winds': 6,
    'Fog or mist': 7,
    'Other': 8,
    'Unknown': 9,
}

road_surface_map = {
    'Dry': 1,
    'Wet/Damp': 2,
    'Snow': 3,
    'Frost/Ice': 4,
    'Flood (Over 3cm of water)': 5,
}

special_conditions_map = {
    'None': 0,
    'Auto traffic singal out': 1,               # typo in data
    'Auto traffic signal out': 1,
    'Auto traffic signal partly defective': 2,
    'Permanent sign or marking defective or obscured': 3,  # variant label
    'Road signs defective or obscured': 3,
    'Roadworks': 4,
    'Road surface defective': 5,
    'Ol or diesel': 6,                          # typo in data
    'Oil or diesel': 6,
    'Mud': 7,
}

carriageway_map = {
    'None': 0,
    'Dislodged vehicle load in carriageway': 1,
    'Other object in carriageway': 2,
    'Involvement with previous accident': 3,
    'Pedestrian in carriageway (not injured)': 6,
    'Any animal (except a ridden horse)': 7,    # variant label
    'Any animal in carriageway (except ridden horse)': 7,
}

junction_control_map = {
    'None': 0,
    'Authorised person': 1,
    'Automatic traffic signal': 2,
    'Stop Sign': 3,
    'Giveway or uncontrolled': 4,
}

ped_human_map = {
    'None within 50 metres': 0,
    'Control by school crossing patrol': 1,
    'Control by other authorised person': 2,
}

ped_physical_map = {
    'No physical crossing within 50 meters': 0,
    'Zebra crossing': 1,
    'non-junction pedestrian crossing': 4,
    'Pedestrian phase at traffic signal junction': 5,
    'Footbridge or subway': 7,
    'Central refuge': 8,
}

police_map = {
    'Yes': 1,
    'No': 2,
    'Unknown': 3,
}

road_type_map = {
    'Roundabout': 1,
    'One way street': 2,
    'Dual carriageway': 3,
    'Single carriageway': 6,
    'Slip road': 7,
    'Unknown': 9,
}

# Apply all mappings
mapping_cols = {
    'Light_Conditions': light_map,
    'Weather_Conditions': weather_map,
    'Road_Surface_Conditions': road_surface_map,
    'Special_Conditions_at_Site': special_conditions_map,
    'Carriageway_Hazards': carriageway_map,
    'Junction_Control': junction_control_map,
    'Pedestrian_Crossing-Human_Control': ped_human_map,
    'Pedestrian_Crossing-Physical_Facilities': ped_physical_map,
    'Did_Police_Officer_Attend_Scene_of_Accident': police_map,
    'Road_Type': road_type_map,
}

for col, mapping in mapping_cols.items():
    df_accident[f"{col}_code"] = df_accident[col].map(mapping)





In [ ]:
# df_ml = df_accident.copy()

In [ ]:
stats_cols = [
    # Target
    "Accident_Severity",

    # Continuous / meaningful numeric
    # "Longitude",
    # "Latitude",
    "Speed_limit",
    "Number_of_Vehicles",
    "Number_of_Casualties",

    # Temporal (raw, not cyclic encoded versions)
    # "Hour",
    # "Day_of_Week",
    # "Month",
    # "Year",

    # Road & environment codes
    "Light_Conditions_code",
    "Weather_Conditions_code",
    "Road_Surface_Conditions_code",
    "Special_Conditions_at_Site_code",
    "Carriageway_Hazards_code",
    "Junction_Control_code",
    "Pedestrian_Crossing-Physical_Facilities_code",
    "Road_Type_code",
    "Urban_or_Rural_Area",
    "1st_Road_Class",
    "2nd_Road_Class",
]

stats = df_accident[stats_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).T
stats = stats.round(2)

stats

Let's set up the features now, we leave out:
- `Did_Police_Officer_Attend_Scene_of_Accident_code`: Police response, does not affect risk
- `Police_Force`: ID, the are similar with locations
- `1st and 2nd Road Number`: They are just ID


In [ ]:
cluster_features = [
    # Geo
    # "Latitude", "Longitude",

    # Risk
    "Accident_Severity",
    "Number_of_Vehicles",
    "Number_of_Casualties",

    # Road info
    "Speed_limit",
    "Urban_or_Rural_Area",
    "1st_Road_Class",
    "2nd_Road_Class",
    "Road_Type_code",
    "Junction_Control_code",

    # Environment
    "Light_Conditions_code",
    "Weather_Conditions_code",
    "Road_Surface_Conditions_code",
    "Special_Conditions_at_Site_code",
    "Carriageway_Hazards_code",

    # Pedestrian
    "Pedestrian_Crossing-Human_Control_code",
    "Pedestrian_Crossing-Physical_Facilities_code",

    # Temporal (cyclical)
    "Hour_sin", "Hour_cos",
    "Day_of_Week_sin", "Day_of_Week_cos",
    "Month_sin", "Month_cos",
    "Year",  # linear

]

df_cluster = df_accident[cluster_features].copy()
print(f"Feature matrix shape: {df_cluster.shape}")  # (1500400, 25)

In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

print(f"Scaled shape: {X_scaled.shape}")

In [ ]:
!pip install kneed

In [ ]:

inertias = []
k_range = range(2, 12)

for k in k_range:
    model = MiniBatchKMeans(n_clusters=k, random_state=8, batch_size=10_000, n_init=3)
    model.fit(X_scaled)
    inertias.append(model.inertia_)
    print(f"k={k} done, inertia={model.inertia_:.0f}")

kl = KneeLocator(k_range, inertias, curve= 'convex', direction= 'decreasing')
optimal_k = kl.elbow

# Plot
plt.figure(figsize=(9, 5))
plt.plot(k_range, inertias, marker='o', linewidth=2, markersize=7)
plt.axvline(optimal_k, color = 'blue', linestyle = ':',
             label= f"Optimal k = {optimal_k}")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method — MiniBatchKMeans")
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Fit final model
kmeans = MiniBatchKMeans(n_clusters=optimal_k, random_state=8, batch_size=10_000, n_init=3)
df_accident["cluster"] = kmeans.fit_predict(X_scaled)

# Check cluster sizes — watch for heavily imbalanced clusters
print(df_accident["cluster"].value_counts().sort_index())

# Profile each cluster
profile = df_accident.groupby("cluster")[cluster_features].mean().round(2)
print(profile.T)  # transpose so features are rows, clusters are columns

In [ ]:
heatmap_features = [f for f in cluster_features if not f.startswith("greater_region")]
profile = df_accident.groupby("cluster")[heatmap_features].mean()
profile_norm = (profile - profile.mean()) / profile.std()

fig, ax = plt.subplots(figsize=(14, 10))

sns.heatmap(
    profile_norm.T,
    ax=ax,
    cmap="RdYlGn_r",
    center=0,
    linewidths=0.4,
    linecolor="#e0e0e0",
    annot=True,
    fmt = '.2f',
    cbar_kws={"shrink": 0.6, "label": "Z-score"},
    xticklabels=[f"C{i}" for i in range(6)],
    yticklabels=heatmap_features,
)
ax.set_title("Cluster Feature Profile — Normalised Z-score", fontsize=13, fontweight="bold", pad=12)
plt.setp(ax.get_xticklabels(), rotation=0)
plt.setp(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
cluster_labels = {
    0: "High-Speed Rural - Fatal Risk",      # 61mph, rural, 2.72 = most fatal skew
    1: "Urban Junction - Signalled",          # urban, signalled, moderate severity
    2: "Roundabout/Mixed - Low Severity",     # 2.91 = mostly slight
    3: "Pedestrian Zone - Patrol",            # school crossing patrol, tiny cluster
    4: "Urban Pedestrian Crossing",           # pelican/puffin crossings
    5: "Urban Non-Junction",                  # no junction, single road
}

df_accident["cluster_label"] = df_accident["cluster"].map(cluster_labels)
print(df_accident["cluster_label"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

cluster_colors = ["#E63946", "#457B9D", "#2A9D8F", "#E9C46A", "#F4A261", "#264653"]
sizes = df_accident["cluster"].value_counts().sort_index()
bars = ax.barh(
    [f"C{i}" for i in sizes.index],
    sizes.values,
    color=cluster_colors,
    edgecolor="none",
    height=0.6,
)
for bar, val in zip(bars, sizes.values):
    ax.text(
        bar.get_width() + 4000, bar.get_y() + bar.get_height() / 2,
        f"{val:,}", va="center", fontsize=9
    )
ax.set_title("Cluster Size Distribution", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Number of Accidents", fontsize=10)
ax.spines[:].set_visible(False)
ax.set_xlim(0, sizes.max() * 1.18)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

metrics  = ["Accident_Severity", "Number_of_Casualties", "Speed_limit"]
m_labels = ["Avg Severity (1=Fatal, 3=Slight)", "Avg Casualties", "Avg Speed Limit"]
m_colors = ["#E63946", "#F4A261", "#457B9D"]
x        = np.arange(6)
width    = 0.22

for i, (metric, label, color) in enumerate(zip(metrics, m_labels, m_colors)):
    vals      = profile[metric].values
    vals_norm = (vals - vals.min()) / (vals.max() - vals.min())
    ax.bar(x + i * width, vals_norm, width, label=label, color=color, alpha=0.85, edgecolor="none")

ax.set_title("Key Metrics per Cluster (Normalised 0–1)", fontsize=13, fontweight="bold", pad=12)
ax.set_xticks(x + width)
ax.set_xticklabels([f"C{i}" for i in range(6)])
ax.set_ylabel("Normalised Value (0–1)", fontsize=10)
ax.legend(fontsize=9, edgecolor="none")
ax.spines[:].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

cross = pd.crosstab(
    df_accident["cluster"],
    df_accident["greater_region"],
    normalize="index"
).round(3) * 100

sns.heatmap(
    cross,
    ax=ax,
    cmap="Blues",
    linewidths=0.4,
    linecolor="#e0e0e0",
    annot=True,
    fmt=".1f",
    cbar_kws={"shrink": 0.5, "label": "% of cluster"},
    xticklabels=cross.columns,
    yticklabels=[f"C{i}" for i in cross.index],
)
ax.set_title("Region Distribution per Cluster (% of cluster row)", fontsize=13, fontweight="bold", pad=12)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.setp(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 🚦 K-Means Clustering Analysis — UK Road Accidents (2005–2016)

### Overview
Using **MiniBatchKMeans** with **k=6** (selected via KneeLocator elbow method),
1,500,400 accident records were segmented into 6 distinct risk profiles based on
23 features spanning road geometry, environmental conditions, pedestrian
infrastructure, and temporal patterns. All features were standardised using
`StandardScaler` prior to clustering, with cyclical encoding applied to `Hour`,
`Day_of_Week`, and `Month` to preserve their circular nature.

---

### Cluster Definitions

| Cluster | Label | Size | Severity (mean) | Casualties (mean) | Speed Limit (mean) |
|---|---|---|---|---|---|
| **C0** | High-Speed Rural — Fatal Risk | 293,600 | **2.72** | **1.66** | **60.8 mph** |
| **C1** | Urban Junction — Signalled | 524,334 | 2.87 | 1.30 | 33.4 mph |
| **C2** | Roundabout / Mixed Junction | 158,299 | 2.91 | 1.29 | 38.8 mph |
| **C3** | Pedestrian Zone — Patrol | 8,845 | 2.87 | 1.26 | 31.4 mph |
| **C4** | Urban Pedestrian Crossing | 194,893 | 2.84 | 1.29 | 31.6 mph |
| **C5** | Urban Non-Junction | 320,429 | 2.86 | 1.21 | 33.0 mph |

---

### Cluster-by-Cluster Interpretation

#### 🔴 C0 — High-Speed Rural (Fatal Risk)
The most dangerous cluster by every metric. With an average speed limit of
**60.8 mph** and `Urban_or_Rural_Area` firmly in rural territory, these accidents
occur on A-roads and Motorways far from urban infrastructure. The heatmap shows
strongly elevated z-scores for `Speed_limit`, `Urban_or_Rural_Area`,
`Number_of_Casualties`, `Light_Conditions_code`, `Road_Surface_Conditions_code`,
`Carriageway_Hazards_code`, and `Special_Conditions_at_Site_code` — indicating
these accidents disproportionately occur under adverse conditions on fast, open
roads. The mean severity of **2.72** (closest to 1=Fatal) makes this the highest
risk cluster, confirmed by the normalised bar chart where C0 scores maximum on
both Speed and Casualties. Geographically, C0 is spread across all regions
(~10% each) with a slight concentration in **South East (18.8%)** and
**East of England (13.7%)**, consistent with high-speed A-road networks there.

#### 🔵 C1 — Urban Junction (Signalled)
The largest cluster at **524,334 accidents (35% of total)**. Characterised by
low speed limits (~33 mph), urban roads, and high `Junction_Control_code`
indicating traffic signal presence. The heatmap shows elevated `2nd_Road_Class`
and `Road_Type_code`, confirming junction-based collisions at intersections.
Severity is mild (2.87 ≈ mostly Slight). **London dominates at 19.7%**,
consistent with its dense signalised junction network. This cluster represents
the typical urban fender-bender scenario — high frequency, low severity.

#### 🟢 C2 — Roundabout / Mixed Junction
The **safest cluster** with severity 2.91 (most Slight). `Road_Type_code`
z-score of **-1.97** is the most extreme value in the entire heatmap, firmly
identifying roundabout geometry as the defining characteristic. Despite moderate
speed limits (~39 mph), the inherent safety design of roundabouts (no
perpendicular collisions) keeps severity low. Scotland (13.2%) and South East
(16.5%) are slightly over-represented. This cluster supports the well-established
road safety evidence that roundabouts reduce fatal collisions compared to
signalised crossroads.

#### 🟡 C3 — Pedestrian Zone (School Patrol)
The **smallest and most distinctive cluster** at just **8,845 accidents (<1%)**.
The defining signal is `Pedestrian_Crossing-Human_Control_code = 2.04` — by far
the highest z-score for this feature — indicating school crossing patrol
presence. Combined with very low speed limits (~31 mph) and urban setting, these
are accidents occurring in active pedestrian management zones, likely near schools
during drop-off/pick-up hours. The temporal features (`Hour_sin = 1.56`,
`Day_of_Week_sin = -1.47`) confirm a distinctive time-of-day and day-of-week
pattern. **North West (17.2%) and Scotland (25.4%)** are notably
over-represented, which may reflect regional school zone enforcement differences.
Despite the pedestrian context, mean severity (2.87) and casualties (1.26) are
not the worst — likely due to low speeds and active supervision.

#### 🟠 C4 — Urban Pedestrian Crossing (Infrastructure)
Defined by the highest `Pedestrian_Crossing-Physical_Facilities_code` z-score
(**1.75**), corresponding to pelican, puffin, and toucan crossings —
signal-controlled pedestrian infrastructure. Unlike C3 (human-controlled), C4
represents accidents near formal signal-controlled crossings in urban areas.
**London is strongly dominant at 27.9%**, reflecting its dense pedestrian
crossing infrastructure. The `Year` z-score of **1.78** is notable — the
highest across all clusters — suggesting these accidents are more concentrated
in later years (post-2011), possibly reflecting increased pedestrian activity
or reporting changes. Severity (2.84) and casualties (1.29) are moderate.

#### 🔷 C5 — Urban Non-Junction
The second largest cluster at **320,429 accidents**. Defined by
`2nd_Road_Class = -0.99` (no second road = not at a junction) and
`Junction_Control_code ≈ 0` (no junction control), these are mid-road,
single-carriageway accidents on urban streets. The lowest average casualties
(1.21) and moderate severity (2.86) suggest these are typically single-vehicle
or rear-end incidents on straight urban roads. The heatmap shows
consistently near-zero z-scores across most features — this is the
"baseline" urban accident profile with no strong environmental or
infrastructure signal. Distribution is relatively even across all regions.

---

### Key Takeaways

1. **Speed is the dominant risk factor** — C0 (rural, 61 mph) has 28% more
   casualties per accident than the next highest cluster and the most fatal skew
   in severity, despite being only the 3rd largest cluster.

2. **Roundabouts work** — C2 is the safest cluster (severity 2.91) despite
   intermediate speeds, supporting their use as a safety intervention.

3. **Urban accidents dominate by volume but not severity** — C1 alone accounts
   for 35% of all accidents, yet has near-lowest danger. High frequency ≠ high risk.

4. **Pedestrian infrastructure creates distinct accident typologies** — C3
   (human patrol) and C4 (signal crossings) are clearly separable, suggesting
   different intervention strategies are needed for each.

5. **Geography matters for pedestrian clusters** — London's dominance in C4
   (27.9%) and Scotland's in C3 (25.4%) suggest regional infrastructure and
   enforcement differences worth investigating further.

6. **Temporal patterns are subtle** — Month seasonality and day-of-week effects
   are present but secondary to road type and speed in driving cluster separation,
   except in C3 where school hours create a distinctive temporal signature.

In [ ]:
df_accident = df_accident[df_accident['greater_region'] != 'England']


In [ ]:
df_accident.info()

In [ ]:
# Check how many are 0
print(df_accident["1st_Road_Number"].value_counts().head(10))

# What road class do the 0s belong to?
print(df_accident[df_accident["1st_Road_Number"] == 0]["1st_Road_Class_Label"].value_counts())

In [ ]:
# Filter out road number 0
df_roads = df_accident[df_accident["1st_Road_Number"] != 0].copy()

# Build clean road_id — e.g. "A40", "M25", "B1234"
df_roads["road_id"] = (
    df_roads["1st_Road_Class_Label"].str.strip() +
    df_roads["1st_Road_Number"].astype(str)
)

# Sanity check
print(f"Rows kept:    {len(df_roads):,}")
print(f"Rows dropped: {len(df_accident) - len(df_roads):,}")
print(f"\nSample road_ids:\n{df_roads['road_id'].value_counts().head(10)}")

In [ ]:
# Fix Motorway labels → M prefix
df_roads["road_id"] = df_roads["road_id"].str.replace("Motorway", "M", regex=False)

# Verify
print(df_roads["road_id"].value_counts().head(10))

In [ ]:
# Aggregate per region + road
hotspots = (
    df_roads.groupby(["greater_region", "road_id"])
    .agg(
        total_accidents  = ("Accident_Index", "count"),
        avg_severity     = ("Accident_Severity", "mean"),
        total_casualties = ("Number_of_Casualties", "sum"),
        fatal_count      = ("Accident_Severity", lambda x: (x == 1).sum()),
    )
    .reset_index()
)

# Danger score — weighted by severity and casualties
hotspots["danger_score"] = (
    hotspots["total_accidents"] *
    (1 / hotspots["avg_severity"]) *
    np.log1p(hotspots["total_casualties"])
).round(2)

# Top 5 per region by danger score
top5 = (
    hotspots.sort_values("danger_score", ascending=False)
    .groupby("greater_region")
    .head(5)
    .sort_values(["greater_region", "danger_score"], ascending=[True, False])
)


In [ ]:
region_colors = {
    "East Midlands":             "#E63946",
    "East of England":           "#457B9D",
    "London":                    "#2A9D8F",
    "North East":                "#E9C46A",
    "North West":                "#F4A261",
    "Scotland":                  "#264653",
    "South East":                "#A8DADC",
    "South West":                "#6A4C93",
    "Wales":                     "#1982C4",
    "West Midlands":             "#8AC926",
    "Yorkshire and The Humber":  "#FF595E",
}

regions_sorted = sorted(top5["greater_region"].unique())

fig, axes = plt.subplots(
    nrows= 2, ncols=6,
    figsize=(20, 8),
    facecolor="white"
)
axes = axes.flatten()

for i, region in enumerate(regions_sorted):
    ax = axes[i]
    df_r = top5[top5["greater_region"] == region].sort_values("danger_score")

    color = region_colors.get(region, "#888888")
    bars = ax.barh(
        df_r["road_id"],
        df_r["danger_score"],
        color=color,
        edgecolor="none",
        height=0.55,
    )

    for bar, (_, row) in zip(bars, df_r.iterrows()):
        ax.text(
            bar.get_width() * 0.98,
            bar.get_y() + bar.get_height() / 2,
            f"{row['total_accidents']:,} acc | {row['fatal_count']} fatal",
            va="center", ha="right",
            color="white", fontsize=8, fontweight="bold"
        )

    region_total = df_roads[df_roads["greater_region"] == region]["Accident_Index"].count()
    ax.set_title(
        f"{region}\n({region_total:,} total accidents)",
        fontsize=10, fontweight="bold", pad=8
    )
    ax.set_xlabel("Danger Score", fontsize=8, color="#555555")
    ax.spines[:].set_visible(False)
    ax.tick_params(colors="#333333", labelsize=9)
    ax.set_xlim(0, df_r["danger_score"].max() * 1.05)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))

axes[-1].set_visible(False)

fig.suptitle(
    "Top 5 Road Hotspots per Greater Region — UK Accidents (2005–2016)\n"
    "Danger Score = Accidents × (1/Avg Severity) × log(Casualties+1)",
    fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:

hotspots_region = (
    df_roads.groupby(["region", "road_id"])
    .agg(
        total_accidents  = ("Accident_Index", "count"),
        avg_severity     = ("Accident_Severity", "mean"),
        total_casualties = ("Number_of_Casualties", "sum"),
        fatal_count      = ("Accident_Severity", lambda x: (x == 1).sum()),
    )
    .reset_index()
)

# Danger score
hotspots_region["danger_score"] = (
    hotspots_region["total_accidents"] *
    (1 / hotspots_region["avg_severity"]) *
    np.log1p(hotspots_region["total_casualties"])
).round(2)


top10_regions = (
    df_roads.groupby("region")["Accident_Index"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

top5_region = (
    hotspots_region.sort_values("danger_score", ascending=False)
    .groupby("region")
    .head(5)
    .sort_values(["region", "danger_score"], ascending=[True, False])
)

top5_top10 = top5_region[top5_region["region"].isin(top10_regions)].copy()

palette = [cm.tab10(i) for i in range(10)]
region_color_map = {r: palette[i] for i, r in enumerate(top10_regions)}

fig, axes = plt.subplots(
    nrows=2, ncols=5,
    figsize=(26, 11),
    facecolor="white"
)
axes = axes.flatten()

for i, region in enumerate(top10_regions):
    ax = axes[i]
    df_r = top5_top10[top5_top10["region"] == region].sort_values("danger_score")

    color = region_color_map.get(region, "#888888")
    bars = ax.barh(
        df_r["road_id"],
        df_r["danger_score"],
        color=color,
        edgecolor="none",
        height=0.55,
    )

    for bar, (_, row) in zip(bars, df_r.iterrows()):
        ax.text(
            bar.get_width() * 0.98,
            bar.get_y() + bar.get_height() / 2,
            f"{row['total_accidents']:,} acc | {row['fatal_count']} fatal",
            va="center", ha="right",
            color="white", fontsize=7.5, fontweight="bold"
        )

    region_total = df_roads[df_roads["region"] == region]["Accident_Index"].count()
    ax.set_title(
        f"{region}\n({region_total:,} total accidents)",
        fontsize=10, fontweight="bold", pad=8
    )
    ax.set_xlabel("Danger Score", fontsize=8, color="#555555")
    ax.spines[:].set_visible(False)
    ax.tick_params(colors="#333333", labelsize=9)
    ax.set_xlim(0, df_r["danger_score"].max() * 1.05)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))

fig.suptitle(
    "Top 5 Road Hotspots — 10 Regions with Most Accidents (2005–2016)\n"
    "Danger Score = Accidents × (1/Avg Severity) × log(Casualties+1)",
    fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

| Cluster | Label | Size | Avg Severity | Avg Casualties | Avg Speed | Key Insight |
|---|---|---|---|---|---|---|
| C0 | High-Speed Rural | 293,600 | 2.72 ⚠️ | 1.66 | 60.8 mph | Rural A-roads & motorways; adverse conditions; highest fatal risk |
| C1 | Urban Junction | 524,334 | 2.87 | 1.30 | 33.4 mph | Largest cluster (35%); signalised junctions; high frequency, low severity |
| C2 | Roundabout / Mixed | 158,299 | 2.91 ✅ | 1.29 | 38.8 mph | Safest cluster; roundabout geometry reduces perpendicular collisions |
| C3 | Pedestrian Patrol | 8,845 | 2.87 | 1.26 | 31.4 mph | Smallest (<1%); school crossing zones; distinctive school-hour timing |
| C4 | Urban Crossing | 194,893 | 2.84 | 1.29 | 31.6 mph | Signal-controlled crossings; London-heavy (27.9%); rising trend post-2011 |
| C5 | Urban Non-Junction | 320,429 | 2.86 | 1.21 | 33.0 mph | Mid-road urban incidents; lowest casualties; no strong environmental signal |

> ⚠️ Highest risk — ✅ Safest cluster · Severity scale: 1 = Fatal, 3 = Slight

- **Speed is the dominant risk factor** — C0 averages 60.8 mph and records 28%
  more casualties per accident than any other cluster, despite being only the
  3rd largest group.
- **Roundabouts demonstrably reduce fatal collisions** — C2 is the safest
  cluster (severity 2.91) despite intermediate speeds of 38.8 mph, supporting
  roundabouts as an effective road safety intervention over signalised crossroads.
- **Geography shapes pedestrian risk** — London accounts for 27.9% of C4
  (signal crossings) while Scotland accounts for 25.4% of C3 (school patrols),
  suggesting regional infrastructure and enforcement differences worth
  investigating.